# Chapter 4: Guidelines and Standard Metrics for Evaluating LLMs
**Module 04 – Introduction to LLMs in Python**

> *Instructor: Iván Palomares Carrascosa, PhD — Senior Data Science & AI Manager*

## 4.1 Why Evaluation Matters

Evaluating LLMs correctly ensures we:
- Measure actual performance (not just training performance)
- Compare models fairly
- Detect overfitting and biases
- Track improvements during fine-tuning

### Classification vs Generation Metrics

| Task | Metrics |
|---|---|
| Classification | Accuracy, Precision, Recall, F1, AUC-ROC |
| Text Generation | BLEU, ROUGE, Perplexity |

## 4.2 Classification Accuracy

In [ ]:
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report

# Load sentiment analysis pipeline
sentiment_analysis = pipeline("sentiment-analysis")

# Test examples with ground truth labels (1=positive, 0=negative)
test_examples = [
    {"text": "I love this product!",               "label": 1},
    {"text": "The service was terrible.",           "label": 0},
    {"text": "This movie is amazing.",              "label": 1},
    {"text": "I'm disappointed with the quality.", "label": 0},
]

# Run inference
predictions = sentiment_analysis([ex["text"] for ex in test_examples])

# Convert to binary labels
true_labels      = [ex["label"] for ex in test_examples]
predicted_labels = [1 if pred["label"] == "POSITIVE" else 0 for pred in predictions]

# Accuracy
acc = accuracy_score(true_labels, predicted_labels)

for ex, pred in zip(test_examples, predicted_labels):
    correct = '✅' if ex['label'] == pred else '❌'
    print(f"{correct} Text: {ex['text']!r:45} → Predicted: {pred}")
print(f"\nAccuracy: {acc:.2%}")

## 4.3 The `evaluate` Library

Hugging Face's `evaluate` library provides standardized metric implementations.

In [ ]:
import evaluate

# Load accuracy metric
accuracy_metric = evaluate.load("accuracy")

# Compute
result = accuracy_metric.compute(
    predictions=predicted_labels,
    references=true_labels
)
print("Accuracy:", result)

# Load and use F1 score
f1_metric = evaluate.load("f1")
f1 = f1_metric.compute(
    predictions=predicted_labels,
    references=true_labels,
    average='binary'
)
print("F1 Score:", f1)

## 4.4 Generation Metrics: BLEU and ROUGE

In [ ]:
# BLEU — measures n-gram overlap between generated and reference text
# Used mainly for translation
bleu = evaluate.load("bleu")
result = bleu.compute(
    predictions=["the cat is on the mat"],
    references=[["the cat is sitting on the mat"]]
)
print("BLEU:", result)

# ROUGE — recall-oriented metric for summarization
rouge = evaluate.load("rouge")
result = rouge.compute(
    predictions=["the cat sat on the mat"],
    references=["the cat is sitting on the mat in the room"]
)
print("\nROUGE scores:")
for k, v in result.items():
    print(f"  {k}: {v:.4f}")

## 4.5 Perplexity — Language Model Metric

Perplexity measures how well a language model **predicts a sample**:

$$PPL = \exp\left(-\frac{1}{N}\sum_{i=1}^{N}\log P(w_i|w_{<i})\right)$$

- **Lower perplexity** = better model (more confident, accurate predictions)
- A perplexity of 1 = perfect prediction

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

tokenizer_ppl = GPT2Tokenizer.from_pretrained('gpt2')
model_ppl = GPT2LMHeadModel.from_pretrained('gpt2')
model_ppl.eval()

def compute_perplexity(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors='pt')
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs['input_ids'])
    return torch.exp(outputs.loss).item()

# Compare perplexity on different texts
natural   = "The weather is beautiful today and I am going for a walk."
artificial = "Sky green the over jumped fox brown the quickly."

ppl_natural    = compute_perplexity(natural, model_ppl, tokenizer_ppl)
ppl_artificial = compute_perplexity(artificial, model_ppl, tokenizer_ppl)

print(f"Natural sentence perplexity:    {ppl_natural:.2f}")
print(f"Artificial sentence perplexity: {ppl_artificial:.2f}")

## Summary

| Metric | Task | Interpretation |
|---|---|---|
| Accuracy | Classification | % correct predictions |
| F1 Score | Classification | Harmonic mean of precision & recall |
| BLEU | Translation | N-gram overlap with reference |
| ROUGE | Summarization | Recall-based overlap |
| Perplexity | Language modeling | Lower = better prediction |

✅ Always **evaluate on held-out test data** — never on training data!